# Assessment 1 — Weeks 1–11

**Time:** 6 hours, open book (your notes, the course notebooks, library documentation) — but your own work: no help from other people or AI assistants.
**Marks:** 100 in eight tasks; the points are given with each task. **Pass:** 60.
**Rules:** every function must run from a fresh kernel in the course environment (`requirements.txt`). Written answers count as much as code: keep them short, precise and in your own words. The test cells check that your results are **consistent** — they do not tell you whether your numbers are right.

**Submit** the notebook, run from top to bottom, for marking. The answer key in `solutions/` is opened only after marking.

| Task | Topic | Module | Points |
|---|---|---|---|
| 1 | Three-phase power and power-factor correction | EE foundations | 8 |
| 2 | Newton's method on a power-flow equation | Maths foundations | 8 |
| 3 | 15-minute data across both daylight-saving changes | M1 | 12 |
| 4 | Market clearing as a welfare LP; a carbon switching price | M2 | 12 |
| 5 | A dispatch LP with a ramp limit: prices, KKT, PyPSA | M3 | 16 |
| 6 | A reservoir and its water value | M3 | 12 |
| 7 | Code review: a forecasting function with leaks | Forecasting | 18 |
| 8 | A newsvendor decision and a brief | Forecasting | 14 |

In [1]:
import logging
import sys
import warnings
from pathlib import Path

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

# the forecasting module's data generator, found from the repository root (works from assessments/ and assessments/solutions/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "COURSE_PLAN.md").exists())
sys.path.insert(0, str(ROOT / "06_energy_forecasting"))
from forecast_data import load_data, temperature_forecast

warnings.filterwarnings("ignore")
logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)


def mae(forecast, actual):
    return float(np.abs(np.asarray(forecast, float) - np.asarray(actual, float)).mean())

## Task 1 — Three-phase power (8 points)

A factory draws **200 kW** at a power factor of **0.90 lagging** from a **400 V** (line-to-line) three-phase supply.

(a) Write `three_phase(P_kW, pf, V_LL)` returning the apparent power S (kVA), the reactive power Q (kvar) and the line current I (A).
(b) Write `capacitor_kvar(P_kW, pf_from, pf_to)`: the capacitor bank (kvar) that raises the power factor from `pf_from` to `pf_to` at constant real power. Compute it for 0.90 → 0.98 and the line current afterwards.
(c) In two sentences: why does the network operator care about the factory's power factor, if the energy (kWh) it buys is unchanged?

In [2]:
def three_phase(P_kW: float, pf: float, V_LL: float) -> dict:
    """Apparent power (kVA), reactive power (kvar) and line current (A) of a balanced three-phase load."""
    ### BEGIN SOLUTION
    S = P_kW / pf
    Q = np.sqrt(S ** 2 - P_kW ** 2)
    I = S * 1e3 / (np.sqrt(3) * V_LL)
    return {"S_kVA": S, "Q_kvar": Q, "I_A": I}
    ### END SOLUTION


def capacitor_kvar(P_kW: float, pf_from: float, pf_to: float) -> float:
    """Capacitor (kvar) that raises the power factor from pf_from to pf_to at constant real power."""
    ### BEGIN SOLUTION
    return P_kW * (np.tan(np.arccos(pf_from)) - np.tan(np.arccos(pf_to)))
    ### END SOLUTION

In [3]:
before = three_phase(200, 0.90, 400)
after = three_phase(200, 0.98, 400)
qc = capacitor_kvar(200, 0.90, 0.98)
assert np.isclose(before["S_kVA"] ** 2, 200 ** 2 + before["Q_kvar"] ** 2), "the power triangle must close"
assert np.isclose(np.sqrt(3) * 400 * before["I_A"] / 1e3, before["S_kVA"]), "S = sqrt(3) V I"
assert np.isclose(before["Q_kvar"] - qc, after["Q_kvar"]), "the capacitor supplies the difference in Q"
print({k: round(v, 1) for k, v in before.items()}, "| capacitor", round(qc, 1), "kvar | current after", round(after["I_A"], 1), "A")

{'S_kVA': 222.2, 'Q_kvar': np.float64(96.9), 'I_A': np.float64(320.8)} | capacitor 56.3 kvar | current after 294.6 A


<!-- BEGIN SOLUTION -->
(a) S = 222.2 kVA, Q = 96.9 kvar, I = 320.8 A. (b) A capacitor bank of 56.3 kvar raises the power factor to 0.98 and lowers the line current to 294.6 A — 8 % less for the same real power.

(c) The network carries **current**, not energy: losses grow with I² and every cable and transformer is sized for the current it must carry, so a poor power factor uses up network capacity and raises losses without delivering a single extra kWh. The reactive power must also be supplied and transported, and it affects voltage — which is why many tariffs charge large customers for reactive energy beyond a threshold.
<!-- END SOLUTION -->

## Task 2 — Newton's method on a power-flow equation (8 points)

Two buses at 1.0 p.u. voltage are joined by a lossless line of susceptance B = 10 p.u. The power sent across it is P = V₁ V₂ B sin δ, where δ is the angle difference.

(a) Write `newton_angle(P, V1=1.0, V2=1.0, B=10.0, tol=1e-10)` that solves for δ (radians) with Newton's method from δ = 0, returning δ and the number of iterations. It must raise a `ValueError` if no solution exists.
(b) Solve for P = 4 p.u. How many iterations does it need?
(c) In two sentences: why can no angle carry P = 12 p.u., and what does that mean physically?

In [4]:
def newton_angle(P: float, V1: float = 1.0, V2: float = 1.0, B: float = 10.0, tol: float = 1e-10) -> tuple[float, int]:
    """Angle difference (rad) that carries P across the line, by Newton's method from 0, and the iterations used."""
    ### BEGIN SOLUTION
    p_max = V1 * V2 * B
    if abs(P) > p_max:
        raise ValueError(f"no solution: |P| = {abs(P)} exceeds the maximum transfer {p_max}")
    delta = 0.0
    for iteration in range(1, 51):
        mismatch = V1 * V2 * B * np.sin(delta) - P
        delta -= mismatch / (V1 * V2 * B * np.cos(delta))
        if abs(V1 * V2 * B * np.sin(delta) - P) < tol:
            return float(delta), iteration
    raise ValueError("Newton's method did not converge")
    ### END SOLUTION

In [5]:
delta, iterations = newton_angle(4.0)
assert abs(10 * np.sin(delta) - 4.0) < 1e-9, "the angle must carry the power"
assert iterations <= 10, "Newton converges in a handful of iterations here"
try:
    newton_angle(12.0)
    raise AssertionError("P = 12 should raise ValueError")
except ValueError:
    pass
print(f"delta = {np.degrees(delta):.3f} degrees after {iterations} iterations")

delta = 23.578 degrees after 4 iterations


<!-- BEGIN SOLUTION -->
(b) δ = 0.4115 rad = 23.58° after 4 iterations — Newton's method converges quadratically: each step roughly doubles the number of correct digits.

(c) The power across the line is V₁V₂B sin δ, and sin δ cannot exceed 1: the most this line can carry is V₁V₂B = 10 p.u., at δ = 90°. Asking for 12 p.u. has no solution — physically, the line has reached its steady-state transfer limit, and near 90° the derivative cos δ goes to zero, so Newton's method would also struggle. Module 04 meets the same limit in full power-flow models, where it shows up as a power flow that does not converge.
<!-- END SOLUTION -->

## Task 3 — 15-minute data across the clock changes (12 points)

`QUARTER_HOURS` (below) holds a site's 15-minute average power in MW, in local time (Europe/Zagreb), for two windows that contain the spring and the autumn clock change of 2025. Three quarter-hours are missing.

(a) Write `clean(series)` returning the series with gaps filled by time interpolation, and the number of quarter-hours that were missing.
(b) Write `hourly_energy(series)` returning energy per hour in MWh.
(c) Write `daily_summary(series)` returning, per local calendar day, the number of hours and the energy in MWh.
(d) In two or three sentences: how many hours do 30 March and 26 October 2025 have, and what goes wrong if the timestamps are first converted to tz-naive local time?

In [6]:
idx = pd.date_range("2025-03-29", "2025-04-01", freq="15min", tz="Europe/Zagreb", inclusive="left").append(
    pd.date_range("2025-10-25", "2025-10-28", freq="15min", tz="Europe/Zagreb", inclusive="left"))
local_hour = idx.hour.to_numpy() + idx.minute.to_numpy() / 60
QUARTER_HOURS = pd.Series(2000 + 300 * np.sin(np.pi * (local_hour - 6) / 12).clip(0), index=idx, name="MW")
QUARTER_HOURS = QUARTER_HOURS.drop(QUARTER_HOURS.index[[40, 41, 300]])        # three quarter-hours lost in transmission

In [7]:
def clean(series: pd.Series) -> tuple[pd.Series, int]:
    """Gap-free 15-minute series (time interpolation) and the number of missing quarter-hours."""
    ### BEGIN SOLUTION
    pieces, missing = [], 0
    for _, part in series.groupby(series.index.date >= pd.Timestamp("2025-06-01").date()):
        full = pd.date_range(part.index[0], part.index[-1], freq="15min")
        missing += len(full) - len(part)
        pieces.append(part.reindex(full).interpolate(method="time"))
    return pd.concat(pieces), missing
    ### END SOLUTION


def hourly_energy(series: pd.Series) -> pd.Series:
    """Energy per hour (MWh) from 15-minute average power (MW)."""
    ### BEGIN SOLUTION
    return (series * 0.25).resample("h").sum(min_count=1).dropna()
    ### END SOLUTION


def daily_summary(series: pd.Series) -> pd.DataFrame:
    """Hours and energy (MWh) per local calendar day."""
    ### BEGIN SOLUTION
    hourly = hourly_energy(series)
    by_day = hourly.groupby(hourly.index.date)
    return pd.DataFrame({"hours": by_day.size(), "MWh": by_day.sum()})
    ### END SOLUTION

In [8]:
filled, n_missing = clean(QUARTER_HOURS)
assert filled.isna().sum() == 0 and str(filled.index.tz) == "Europe/Zagreb"
energy = hourly_energy(filled)
assert np.isclose((filled * 0.25).sum(), energy.sum()), "resampling must conserve energy"
summary = daily_summary(filled)
assert np.isclose(summary.MWh.sum(), energy.sum())
print("missing quarter-hours:", n_missing)
summary.round(1)

missing quarter-hours: 3


            hours      MWh
2025-03-29     24  50290.4
2025-03-30     23  48291.0
2025-03-31     24  50291.0
2025-10-25     24  50291.0
2025-10-26     25  52291.0
2025-10-27     24  50291.0

<!-- BEGIN SOLUTION -->
(a) Three quarter-hours were missing and are filled by time interpolation.

(d) On **30 March 2025** clocks jump from 02:00 to 03:00, so the local day has **23 hours**; on **26 October 2025** they fall back from 03:00 to 02:00 and the day has **25 hours** — visible in the summary, where those days carry one hour less or more energy. If the timestamps are made tz-naive, the October hour from 02:00 to 03:00 appears twice with identical labels: resampling merges the two hours into one, doubling that hour's energy or losing one when duplicates are dropped; and in March the missing hour looks like a data gap that an interpolation would wrongly fill with an hour that never existed. Daily energy is then wrong by about one hour's consumption (~2 GWh here). Keeping the index tz-aware — or working in UTC and converting only for display — avoids both.
<!-- END SOLUTION -->

## Task 4 — Market clearing (12 points)

(a) Clear the hour below as a **welfare-maximising LP** in Linopy (M2.1): maximise the value of accepted demand bids minus the cost of accepted supply offers, subject to supply = demand. Write `clear_market(supply, demand)` returning the clearing price (€/MWh, read from the dual — mind the sign), the accepted quantities and the welfare (€).
(b) A coal plant (fuel 14 €/MWh_th, efficiency 0.38, 0.34 t CO₂/MWh_th) competes with a gas plant (fuel 30 €/MWh_th, efficiency 0.55, 0.20 t CO₂/MWh_th). Write `switching_price(...)`: the CO₂ price at which their marginal costs are equal.
(c) In two sentences: who sets the price in (a), and why is that consistent with the accepted and rejected bids?

In [9]:
SUPPLY = pd.DataFrame({"MW": [300, 350, 400, 500, 300, 200], "EUR_per_MWh": [0, 12, 45, 89, 94, 129]},
                      index=pd.Index(["solar", "nuclear", "hydro", "ccgt", "coal", "ocgt"], name="offer"))
DEMAND = pd.DataFrame({"MW": [1200, 200, 150, 100], "EUR_per_MWh": [3000, 100, 80, 60]},
                      index=pd.Index(["inflexible", "industry_a", "industry_b", "industry_c"], name="bid"))

In [10]:
def clear_market(supply: pd.DataFrame, demand: pd.DataFrame) -> dict:
    """Clearing price (EUR/MWh, positive), accepted supply and demand (MW) and welfare (EUR) of one hour."""
    ### BEGIN SOLUTION
    m = linopy.Model()
    s = m.add_variables(lower=0, upper=xr.DataArray(supply.MW.to_numpy(), coords=[supply.index]), name="supply")
    d = m.add_variables(lower=0, upper=xr.DataArray(demand.MW.to_numpy(), coords=[demand.index]), name="demand")
    m.add_constraints(s.sum() - d.sum() == 0, name="balance")
    value = (d * xr.DataArray(demand.EUR_per_MWh.to_numpy(), coords=[demand.index])).sum()
    cost = (s * xr.DataArray(supply.EUR_per_MWh.to_numpy(), coords=[supply.index])).sum()
    m.add_objective(value - cost, sense="max")
    m.solve(solver_name="highs", output_flag=False)
    price = abs(float(m.constraints["balance"].dual))          # a maximisation flips the dual's sign
    return {"price": price, "supply": s.solution.to_pandas(), "demand": d.solution.to_pandas(),
            "welfare": float(m.objective.value)}
    ### END SOLUTION


def switching_price(fuel_a, eff_a, ef_a, fuel_b, eff_b, ef_b) -> float:
    """CO2 price (EUR/t) at which plants a and b have equal marginal cost (fuel EUR/MWh_th, t CO2/MWh_th)."""
    ### BEGIN SOLUTION
    return (fuel_b / eff_b - fuel_a / eff_a) / (ef_a / eff_a - ef_b / eff_b)
    ### END SOLUTION

In [11]:
result = clear_market(SUPPLY, DEMAND)
p = result["price"]
assert np.isclose(result["supply"].sum(), result["demand"].sum())
assert p > 0, "a clearing price in this market is positive - check the sign of the dual"
assert (SUPPLY.EUR_per_MWh[result["supply"] > 1e-6] <= p + 1e-6).all(), "accepted offers are at or below the price"
assert (DEMAND.EUR_per_MWh[result["demand"] > 1e-6] >= p - 1e-6).all(), "accepted bids are at or above the price"
co2 = switching_price(14, 0.38, 0.34, 30, 0.55, 0.20)
assert np.isclose((14 + co2 * 0.34) / 0.38, (30 + co2 * 0.20) / 0.55)
print(f"price {p:.2f} EUR/MWh, volume {result['demand'].sum():.0f} MW, welfare {result['welfare']:,.0f} EUR; "
      f"switching price {co2:.2f} EUR/t")

price 89.00 EUR/MWh, volume 1400 MW, welfare 3,566,650 EUR; switching price 33.33 EUR/t


<!-- BEGIN SOLUTION -->
(a) The market clears at **89 €/MWh** with 1,400 MW traded; welfare is 3,566,650 € (dominated by the inflexible demand's 3,000 €/MWh valuation, which acts like a price cap). (b) The switching price is **33.33 €/t**: both plants then cost 66.67 €/MWh, and below it coal is cheaper.

(c) The **CCGT** sets the price: it is the only partly accepted offer (350 of 500 MW). Everything offered below 89 €/MWh is accepted (solar, nuclear, hydro) and coal at 94 €/MWh is rejected; on the demand side the inflexible load and industry_a (100 €/MWh) are accepted, while industry_b (80) and industry_c (60) are rejected because they value electricity below the price. The dual of the balance constraint is −89 in the solver's sign convention for a maximisation; the price is its absolute value — the sign trap of M2.1.
<!-- END SOLUTION -->

## Task 5 — A dispatch LP with a ramp limit (16 points)

Demand is **400, 900 and 500 MW** in three consecutive hours. A base-load plant (800 MW, 20 €/MWh) can change its output by at most **200 MW per hour**; a peaker (500 MW, 100 €/MWh) has no ramp limit.

(a) Write `dispatch()`: the cost-minimising LP in Linopy, returning the cost, both plants' output and the three hourly prices.
(b) Reproduce it in PyPSA (`pypsa_dispatch()`) and show that cost and prices agree to 10⁻⁶.
(c) One hour has a **negative** price although both plants have positive costs. Explain it with the KKT conditions in three or four sentences: which constraint binds, what is its shadow price, and how do the two add up to the price?

In [12]:
DEMAND_MW = np.array([400.0, 900.0, 500.0])


def dispatch() -> dict:
    """Cost (EUR), output of 'base' and 'peaker' (MW) and hourly prices (EUR/MWh) of the 3-hour LP."""
    ### BEGIN SOLUTION
    t = pd.RangeIndex(3, name="t")
    m = linopy.Model()
    base = m.add_variables(lower=0, upper=800, coords=[t], name="base")
    peak = m.add_variables(lower=0, upper=500, coords=[t], name="peaker")
    later = xr.DataArray(np.arange(3) > 0, coords=[t])
    m.add_constraints(base - base.shift(t=1) <= 200, name="ramp_up", mask=later)
    m.add_constraints(base.shift(t=1) - base <= 200, name="ramp_down", mask=later)
    m.add_constraints(base + peak == xr.DataArray(DEMAND_MW, coords=[t]), name="balance")
    m.add_objective(20 * base.sum() + 100 * peak.sum())
    m.solve(solver_name="highs", output_flag=False)
    return {"cost": float(m.objective.value), "base": base.solution.to_numpy(), "peaker": peak.solution.to_numpy(),
            "price": m.constraints["balance"].dual.to_numpy() + 0.0,
            "ramp_up_dual": m.constraints["ramp_up"].dual.to_numpy()}
    ### END SOLUTION


def pypsa_dispatch() -> pypsa.Network:
    """The same 3-hour dispatch in PyPSA, optimised."""
    ### BEGIN SOLUTION
    n = pypsa.Network()
    n.set_snapshots(range(3))
    n.add("Bus", "bus")
    n.add("Load", "load", bus="bus", p_set=DEMAND_MW)
    n.add("Generator", "base", bus="bus", p_nom=800, marginal_cost=20, ramp_limit_up=200 / 800, ramp_limit_down=200 / 800)
    n.add("Generator", "peaker", bus="bus", p_nom=500, marginal_cost=100)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n
    ### END SOLUTION

In [13]:
own = dispatch()
n = pypsa_dispatch()
assert np.allclose(own["base"] + own["peaker"], DEMAND_MW)
assert np.isclose(n.objective, own["cost"], rtol=1e-6)
assert np.abs(n.buses_t.marginal_price.iloc[:, 0].to_numpy() - own["price"]).max() < 1e-6
assert np.all(np.abs(np.diff(own["base"])) <= 200 + 1e-6)
print(f"cost {own['cost']:,.0f} EUR | base {own['base']} | peaker {own['peaker']} | prices {own['price']}")

cost 60,000 EUR | base [400. 600. 500.] | peaker [  0. 300.   0.] | prices [-60. 100.  20.]


<!-- BEGIN SOLUTION -->
(a, b) Cost 60,000 €; base 400 / 600 / 500 MW, peaker 0 / 300 / 0 MW; prices −60, 100 and 20 €/MWh — PyPSA gives the same cost and prices to 10⁻⁶.

(c) In hour 1 the base plant needs to reach 900 MW of demand but can only climb 200 MW from its 400 MW in hour 0, so the **ramp-up constraint between hours 0 and 1 binds**; its shadow price is **−80 €/MWh** — one more MW of ramp would let base replace one MWh of the 100 €/MWh peaker with 20 €/MWh energy. The base plant runs strictly between its limits in hour 0, so KKT gives price₀ = its marginal cost plus the ramp term: 20 − 80 = **−60 €/MWh**. Read as a story: one more MWh of demand in hour 0 costs 20 € of base fuel but lets the base plant start hour 1 one MW higher, saving 80 € of peaker fuel there — the system is better off by 60 €, so the price is negative. Hour 1's price is the peaker's 100 €/MWh; in hour 2 the ramp-down from 600 to 500 MW does not bind, and base sets the price at 20 €/MWh.
<!-- END SOLUTION -->

## Task 6 — A reservoir and its water value (12 points)

A hydro plant with a 100 MW turbine may release **550 MWh** of water over the day `PRICES` below (€/MWh); it is a price-taker and has no other costs.

(a) Write `reservoir()`: the revenue-maximising LP, returning the dispatch, the revenue and the **water value** (the dual of the energy constraint, as a positive €/MWh).
(b) Check the KKT pattern: in every hour the turbine runs at full power the price is at least the water value; where it is off, at most.
(c) In three sentences: why would offering the water at 0 €/MWh in the day-ahead market be a mistake, and what should the plant bid instead?

In [14]:
PRICES = pd.Series([62, 58, 55, 54, 56, 63, 78, 95, 102, 98, 88, 80, 74, 70, 72, 79, 90, 108, 121, 115, 104, 92, 81, 70],
                   index=pd.RangeIndex(24, name="hour"), dtype=float)


def reservoir() -> dict:
    """Dispatch (MW), revenue (EUR) and water value (EUR/MWh, positive) of the 550 MWh day."""
    ### BEGIN SOLUTION
    m = linopy.Model()
    g = m.add_variables(lower=0, upper=100, coords=[PRICES.index], name="turbine")
    m.add_constraints(g.sum() == 550, name="energy")
    m.add_objective((g * xr.DataArray(PRICES.to_numpy(), coords=[PRICES.index])).sum(), sense="max")
    m.solve(solver_name="highs", output_flag=False)
    return {"dispatch": g.solution.to_pandas(), "revenue": float(m.objective.value),
            "water_value": abs(float(m.constraints["energy"].dual))}
    ### END SOLUTION

In [15]:
res = reservoir()
g, wv = res["dispatch"], res["water_value"]
assert np.isclose(g.sum(), 550) and wv > 0
assert (PRICES[g > 100 - 1e-6] >= wv - 1e-6).all(), "full power only where the price is at least the water value"
assert (PRICES[g < 1e-6] <= wv + 1e-6).all(), "off only where the price is at most the water value"
print(f"revenue {res['revenue']:,.0f} EUR, water value {wv:.1f} EUR/MWh, running in hours {list(g[g > 1e-6].index)}")

revenue 59,900 EUR, water value 98.0 EUR/MWh, running in hours [8, 9, 17, 18, 19, 20]


<!-- BEGIN SOLUTION -->
(a) Revenue 59,900 €; the turbine runs at full power in hours 8, 17, 18, 19 and 20 (102–121 €/MWh) and at 50 MW in hour 9 (98 €/MWh). The **water value is 98 €/MWh** — the price of the partly used hour, since one more MWh of water would be sold there. (b) The KKT pattern holds: every full-power hour is priced at or above 98 €/MWh, every idle hour at or below.

(c) Water has an **opportunity cost**: a MWh released now cannot be released at the evening peak. Offered at 0 €/MWh it would be dispatched first in every hour, so the reservoir would spend its 550 MWh in the night and morning at 54–63 €/MWh instead of 98–121 €/MWh in the peak — losing much of its revenue and leaving the peak to dearer plants. It should bid at its water value, about 98 €/MWh, so that it runs exactly when the price exceeds what the water is worth later (Module 03, lesson 7).
<!-- END SOLUTION -->

## Task 7 — Code review: a forecasting function with leaks (18 points)

A colleague sends you this day-ahead demand forecast and reports an excellent MAE. The data are the forecasting module's (`load_data`, `temperature_forecast`); the forecast is for the day-ahead market, issued at noon on D−1.

(a) List **three** ways in which `flawed_forecast_mae` uses information it would not have, and for each say what it does to the reported MAE.
(b) Write `honest_forecast_mae()`: a corrected version — split in time (train on 2023–2024, test on 2025), use only features known at noon on D−1, the temperature **forecast**, holidays treated like Sundays — and report its MAE on 2025.
(c) In two sentences: what would you tell your colleague about the reported number?

In [16]:
DATA = load_data()
DATA["temperature_forecast"] = temperature_forecast(DATA)


def flawed_forecast_mae() -> float:
    """A colleague's day-ahead forecast. Do not change this function."""
    d = DATA
    X = pd.DataFrame({"hour": d.index.hour, "weekday": d.index.dayofweek, "day_of_year": d.index.dayofyear,
                      "temperature": d.temperature, "demand_1h_ago": d.demand.shift(1)}).dropna()
    y = d.demand.loc[X.index]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    model = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(X_train, y_train)
    return mae(model.predict(X_test), y_test)


print(f"reported MAE: {flawed_forecast_mae():.1f} MW")

reported MAE: 26.4 MW


<!-- BEGIN SOLUTION -->
(a) Three leaks:

1. **Shuffled split.** `train_test_split` draws test hours at random, so every test hour has its neighbours — almost identical, because demand persists — in the training data. The model is tested on what it has effectively seen: the MAE is far too low.
2. **`demand_1h_ago`.** A day-ahead forecast issued at noon on D−1 is 13 to 36 hours ahead; demand one hour before the target hour is never known. The model learns to copy it, which is most of its apparent accuracy.
3. **Actual temperature.** At noon on D−1 only a temperature forecast exists; using the observed temperature removes the weather-forecast error — lesson 2's weather leak, worth about 20 % of the MAE there.

Also worth flagging, though not a leak: `weekday` ignores holidays, which makes the forecast worse on those days, not better.
<!-- END SOLUTION -->

In [17]:
def honest_forecast_mae() -> float:
    """MAE (MW) on 2025 of a day-ahead forecast that uses only information available at noon on D-1."""
    ### BEGIN SOLUTION
    d, idx = DATA, DATA.index
    dow = idx.dayofweek
    daytype = np.where(d.holiday | (dow == 6), 2, np.where(dow == 5, 1, 0))
    issue = idx.normalize() - pd.Timedelta(hours=13)                   # 11:00 on D-1, the last known hour
    X = pd.DataFrame({"hour": idx.hour, "daytype": daytype, "day_of_year": idx.dayofyear,
                      "temperature_forecast": d.temperature_forecast,
                      "last_known": d.demand.reindex(issue).to_numpy(),
                      "same_hour_2_days_ago": d.demand.shift(48), "same_hour_last_week": d.demand.shift(168)},
                     index=idx).dropna()
    y = d.demand.loc[X.index]
    train = X.index < "2025"
    model = HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05, random_state=0).fit(X[train], y[train])
    return mae(model.predict(X[~train]), y[~train])
    ### END SOLUTION

In [18]:
flawed, honest_ = flawed_forecast_mae(), honest_forecast_mae()
assert honest_ > 1.5 * flawed, "an honest day-ahead forecast cannot match a leaky one"
print(f"reported {flawed:.1f} MW, honest {honest_:.1f} MW")

reported 26.4 MW, honest 76.9 MW


<!-- BEGIN SOLUTION -->
(c) The reported 26.4 MW is not a forecast anyone could deliver: it rests on information that does not exist at noon on D−1, tested on hours the model has effectively seen. Measured honestly — trained on 2023–2024, tested on 2025, with only features known at issue time and the temperature forecast — the same kind of model reaches 76.9 MW, about three times worse, which is still a good forecast against the roughly 140 MW of a weekly baseline; that is the number to report and to compare with the TSO's forecast.
<!-- END SOLUTION -->

## Task 8 — A newsvendor decision and a brief (14 points)

A supplier buys its day-ahead demand forecast. On its sunny-weekend portfolio a **surplus** is expensive — it must be sold back at deeply negative prices, costing **80 €/MWh** — while a **shortfall** costs only **20 €/MWh**. `ERRORS_2024` are honest past errors (actual − forecast, MW) of its forecast `FORECAST_2025`, with `ACTUAL_2025` the outcome.

(a) Write `cost(schedule, short, long)` and `best_alpha(short, long)`. Which quantile should the supplier buy, and should it buy more or less than the forecast?
(b) Compare the 2025 cost of buying the forecast with buying the forecast plus the α*-quantile of `ERRORS_2024`.
(c) Write a brief of at most 120 words to the supplier's commercial manager: the recommendation, the saving, and one risk of the approach.

In [19]:
rng = np.random.default_rng(42)
FORECAST_2025 = pd.Series(1000 + 200 * np.sin(np.arange(8760) * 2 * np.pi / 24), name="MW")
ERRORS_2024 = pd.Series(rng.normal(0, 60, 8784))
ACTUAL_2025 = FORECAST_2025 + rng.normal(0, 60, 8760)

In [20]:
def cost(schedule, short: float, long: float, actual=ACTUAL_2025) -> float:
    """Imbalance cost (EUR) of buying `schedule` when `actual` is consumed."""
    ### BEGIN SOLUTION
    gap = np.asarray(actual) - np.asarray(schedule)
    return float((short * np.clip(gap, 0, None) + long * np.clip(-gap, 0, None)).sum())
    ### END SOLUTION


def best_alpha(short: float, long: float) -> float:
    """Cost-minimising quantile to buy."""
    ### BEGIN SOLUTION
    return short / (short + long)
    ### END SOLUTION

In [21]:
alpha = best_alpha(20, 80)
schedule = FORECAST_2025 + np.quantile(ERRORS_2024, alpha)
point, better = cost(FORECAST_2025, 20, 80), cost(schedule, 20, 80)
assert better < point, "buying the right quantile must be cheaper here"
assert np.isclose(best_alpha(50, 50), 0.5)
print(f"alpha* = {alpha:.2f}, shift {np.quantile(ERRORS_2024, alpha):+.1f} MW: "
      f"cost {point / 1e6:.3f} -> {better / 1e6:.3f} MEUR ({100 * (1 - better / point):.1f} % saved)")

alpha* = 0.20, shift -50.3 MW: cost 21.103 -> 14.731 MEUR (30.2 % saved)


<!-- BEGIN SOLUTION -->
(a) α* = 20 / (20 + 80) = **0.2**: when surpluses cost four times as much as shortfalls, buy **less** than the forecast — here the 20 % quantile, 50.3 MW below it. (b) The 2025 imbalance cost falls from 21.10 to 14.73 M€, **30 % less**.

(c) A possible brief:

> **Recommendation:** on the sunny-weekend portfolio, buy about 50 MW less than the demand forecast in every hour. Surpluses currently cost us 80 €/MWh against 20 €/MWh for shortfalls, so slightly under-buying is cheaper on balance. **Saving:** on 2025 data this cuts imbalance costs from 21.1 to 14.7 M€, about 6.4 M€ (30 %). **Risk:** the rule depends on today's cost ratio and on our forecast errors staying as they were in 2024; if shortfall prices spike — in a scarcity event, say — under-buying becomes expensive. We should review the shift monthly and cap it during expected tight periods.
<!-- END SOLUTION -->